# What does this Script do?

## Imports the Relevant data from teh Partners in care Looker database, cleans and combines it, then updates a SQl table where its used to maintain a PowerBI Dashboard.  

import python packages

In [ ]:
import pandas as pd
import looker_sdk
from io import StringIO

In [ ]:
# Load the Looker Python SDK tools.
import looker_sdk

# Path to the `looker.ini` file we setup.


CREDENTIALS_PATH = "utils\\looker.ini"  # <--- Change


# Initializing the Looker SDK.


sdk = looker_sdk.init40(CREDENTIALS_PATH)

# Retrieve my user information from Looker.
my_user = sdk.me()

# Display my username.
my_user["display_name"]

LOS table pulling 

In [ ]:
#############
# Parameters
#############
RESULT_FORMAT = "csv"

#############
# Send Query
#############

# Define your query
body = {
    "model": "oahu_connection_model",
    "view": "base",
    "fields": [
        "agencies.name",
        "agencies.victim_service_provider",
        "clients.count",
        "enrollments.average_duration",
        "enrollments.count",
        "primary_sites.coc",
        "programs.continuum_project",
        "programs.name",
        "programs.project_type_code",
        "programs.availability_start_date",
        "programs.availability_end_date",
        "programs.status",
        "programs.id",
        "enrollments.total_days_in_project_during_reporting_period",
        "programs.ref_target_b",
        "program_hmis_participation_statuses.hmis_participation_status",
        "program_inventory.total_bed_inventory",
        "enrollments.total_days_in_project_during_reporting_period",
        "program_inventory_hmis.total_bed_inventory_ap",
    ],
    "filters": {
        "enrollments.date_filter": "1 quarter",
        "program_inventory.end_date": "null",
    },
    "limit": 100000,
}

################
# Get Response
################

# Run the inline query
result = sdk.run_inline_query(result_format=RESULT_FORMAT, body=body)

# Convert to dataframe.
program_level_df = pd.read_csv(StringIO(result))

In [ ]:
program_level_df[program_level_df["Programs Program ID"].duplicated(keep=False)]

In [ ]:
#############
# Parameters
#############
RESULT_FORMAT = "csv"

#############
# Send Query
#############

# Define your query
body = {
    "model": "oahu_connection_model",
    "view": "base",
    "fields": [
        "programs.id",
        "enrollments.average_duration",
        "clients.count",
    ],
    "filters": {
        "enrollments.date_filter": "1 quarter",
        "program_inventory.end_date": "null",
        "household_move_in_date.move_in_date": "null",
    },
    "limit": 100000,
}

################
# Get Response
################

# Run the inline query
result = sdk.run_inline_query(result_format=RESULT_FORMAT, body=body)

# Convert to dataframe.
counts_without_move_in_date = pd.read_csv(StringIO(result))

In [ ]:
counts_without_move_in_date.rename(
    columns={
        "Enrollments Average Days in Project": "Enrollments Average Days in Project Without Move In",
        "Clients Count Client IDs": "Clients Enrolled During the Reporting Period Without Move In",
    },
    inplace=True,
)

In [ ]:
#############
# Parameters
#############
RESULT_FORMAT = "csv"

#############
# Send Query
#############

# Define your query
body = {
    "model": "oahu_connection_model",
    "view": "base",
    "fields": [
        "programs.id",
        "enrollments.average_duration",
        "clients.count",
    ],
    "filters": {
        "enrollments.date_filter": "1 quarter",
        "program_inventory.end_date": "null",
        "household_move_in_date.move_in_date": "not null",
    },
    "limit": 100000,
}

################
# Get Response
################

# Run the inline query
result = sdk.run_inline_query(result_format=RESULT_FORMAT, body=body)

# Convert to dataframe.
counts_with_move_in_date = pd.read_csv(StringIO(result))

In [ ]:
counts_with_move_in_date.rename(
    columns={
        "Enrollments Average Days in Project": "Enrollments Average Days in Project With Move In",
        "Clients Count Client IDs": "Clients Enrolled During the Reporting Period With Move In",
    },
    inplace=True,
)

In [ ]:
program_level_df = program_level_df.merge(
    counts_with_move_in_date,
    how="left",
    on="Programs Program ID",
)

In [ ]:
program_level_df = program_level_df.merge(
    counts_without_move_in_date,
    how="left",
    on="Programs Program ID",
)

In [ ]:
program_level_df[program_level_df["Programs Program ID"].duplicated(keep=False)]

In [ ]:
import numpy as np

In [ ]:
program_level_df["Program Utilization During The Reporting Period"] = np.where(
    program_level_df["Bed and Unit Inventory Total Bed Inventory for Reporting Period"],
    program_level_df["Enrollments Total Days in Project During the Reporting Period"]
    / program_level_df[
        "Bed and Unit Inventory Total Bed Inventory for Reporting Period"
    ],
    np.nan,
).round(4)

In [ ]:
body = {
    "model": "oahu_connection_model",
    "view": "base",
    "fields": [
        "programs.name",
        "programs.id",
        "enrollments.id",
        "enrollments.start_date",
        "household_move_in_date.move_in_date",
    ],
    "filters": {
        "enrollments.date_filter": "1 quarter",
        "enrollments.still_in_program": "Yes",
    },
    "limit": 100000,
}

################
# Get Response
################

# Run the inline query
result = sdk.run_inline_query(result_format=RESULT_FORMAT, body=body)

# Convert to dataframe.
enrollment_level_df = pd.read_csv(StringIO(result))

In [ ]:
from datetime import datetime

In [ ]:
enrollment_level_df["Enrollments Project Start Date"] = pd.to_datetime(
    enrollment_level_df["Enrollments Project Start Date"]
).copy()

In [ ]:
enrollment_level_df["Days Since Start"] = (
    datetime.now() - enrollment_level_df["Enrollments Project Start Date"]
).dt.days.astype(int)

In [ ]:
enrollment_level_df["day_category_90"] = enrollment_level_df["Days Since Start"].apply(
    lambda x: "over 90 days" if x > 90 else "less than 90 days"
)

enrollment_level_df["day_category_180"] = enrollment_level_df["Days Since Start"].apply(
    lambda x: "over 180 days" if x > 180 else "less than 180 days"
)

In [ ]:
conditions = {
    "Active Enrollments Open For Less Than 90 Days With Move In Date": (
        enrollment_level_df["day_category_90"] == "less than 90 days"
    )
    & enrollment_level_df["Enrollments Household Move-In Date"].notna(),
    "Active Enrollments Open For Less Than 90 Days Without Move In Date": (
        enrollment_level_df["day_category_90"] == "less than 90 days"
    )
    & enrollment_level_df["Enrollments Household Move-In Date"].isna(),
    "Active Enrollments Open For More Than 90 Days With Move In Date": (
        enrollment_level_df["day_category_90"] == "over 90 days"
    )
    & enrollment_level_df["Enrollments Household Move-In Date"].notna(),
    "Active Enrollments Open For More Than 90 Days Without Move In Date": (
        enrollment_level_df["day_category_90"] == "over 90 days"
    )
    & enrollment_level_df["Enrollments Household Move-In Date"].isna(),
    "Active Enrollments Open For Less Than 180 Days With Move In Date": (
        enrollment_level_df["day_category_180"] == "less than 180 days"
    )
    & enrollment_level_df["Enrollments Household Move-In Date"].notna(),
    "Active Enrollments Open For Less Than 180 Days Without Move In Date": (
        enrollment_level_df["day_category_180"] == "less than 180 days"
    )
    & enrollment_level_df["Enrollments Household Move-In Date"].isna(),
    "Active Enrollments Open For More Than 180 Days With Move In Date": (
        enrollment_level_df["day_category_180"] == "over 180 days"
    )
    & enrollment_level_df["Enrollments Household Move-In Date"].notna(),
    "Active Enrollments Open For More Than 180 Days Without Move In Date": (
        enrollment_level_df["day_category_180"] == "over 180 days"
    )
    & enrollment_level_df["Enrollments Household Move-In Date"].isna(),
}

# Initialize a dictionary to store results
results = {}

# Iterate over conditions and apply groupby with size to count occurrences
for key, condition in conditions.items():
    results[key] = enrollment_level_df[condition].groupby("Programs Program ID").size()

# Convert results to a DataFrame
result_df = pd.DataFrame(results).fillna(0).astype(int).reset_index()

In [ ]:
merge_df = pd.merge(
    program_level_df,
    result_df,
    how="left",
    on="Programs Program ID",
)

In [ ]:
merge_df[
    [
        "Active Enrollments Open For Less Than 90 Days With Move In Date",
        "Active Enrollments Open For Less Than 90 Days Without Move In Date",
        "Active Enrollments Open For More Than 90 Days With Move In Date",
        "Active Enrollments Open For More Than 90 Days Without Move In Date",
        "Active Enrollments Open For Less Than 180 Days With Move In Date",
        "Active Enrollments Open For Less Than 180 Days Without Move In Date",
        "Active Enrollments Open For More Than 180 Days With Move In Date",
        "Active Enrollments Open For More Than 180 Days Without Move In Date",
        "Enrollments Average Days in Project With Move In",
        "Clients Enrolled During the Reporting Period With Move In",
        "Enrollments Average Days in Project Without Move In",
        "Clients Enrolled During the Reporting Period Without Move In",
    ]
] = (
    merge_df[
        [
            "Active Enrollments Open For Less Than 90 Days With Move In Date",
            "Active Enrollments Open For Less Than 90 Days Without Move In Date",
            "Active Enrollments Open For More Than 90 Days With Move In Date",
            "Active Enrollments Open For More Than 90 Days Without Move In Date",
            "Active Enrollments Open For Less Than 180 Days With Move In Date",
            "Active Enrollments Open For Less Than 180 Days Without Move In Date",
            "Active Enrollments Open For More Than 180 Days With Move In Date",
            "Active Enrollments Open For More Than 180 Days Without Move In Date",
            "Enrollments Average Days in Project With Move In",
            "Clients Enrolled During the Reporting Period With Move In",
            "Enrollments Average Days in Project Without Move In",
            "Clients Enrolled During the Reporting Period Without Move In",
        ]
    ]
    .fillna(0)
    .copy()
)

In [ ]:
merge_df[
    [
        "Active Enrollments Open For Less Than 90 Days With Move In Date",
        "Active Enrollments Open For Less Than 90 Days Without Move In Date",
        "Active Enrollments Open For More Than 90 Days With Move In Date",
        "Active Enrollments Open For More Than 90 Days Without Move In Date",
        "Active Enrollments Open For Less Than 180 Days With Move In Date",
        "Active Enrollments Open For Less Than 180 Days Without Move In Date",
        "Active Enrollments Open For More Than 180 Days With Move In Date",
        "Active Enrollments Open For More Than 180 Days Without Move In Date",
        "Enrollments Average Days in Project With Move In",
        "Clients Enrolled During the Reporting Period With Move In",
        "Enrollments Average Days in Project Without Move In",
        "Clients Enrolled During the Reporting Period Without Move In",
    ]
] = (
    merge_df[
        [
            "Active Enrollments Open For Less Than 90 Days With Move In Date",
            "Active Enrollments Open For Less Than 90 Days Without Move In Date",
            "Active Enrollments Open For More Than 90 Days With Move In Date",
            "Active Enrollments Open For More Than 90 Days Without Move In Date",
            "Active Enrollments Open For Less Than 180 Days With Move In Date",
            "Active Enrollments Open For Less Than 180 Days Without Move In Date",
            "Active Enrollments Open For More Than 180 Days With Move In Date",
            "Active Enrollments Open For More Than 180 Days Without Move In Date",
            "Enrollments Average Days in Project With Move In",
            "Clients Enrolled During the Reporting Period With Move In",
            "Enrollments Average Days in Project Without Move In",
            "Clients Enrolled During the Reporting Period Without Move In",
        ]
    ]
    .astype(int)
    .copy()
)

In [ ]:
merge_df.rename(
    columns={
        "Clients Count Client IDs": "Clients Enrolled During the Reporting Period",
        "Enrollments Count": "Total Enrollments During the Reporting Period",
    },
    inplace=True,
)

In [ ]:
merge_df

In [ ]:
from ds_sql_connections import GsAutomationReliefAssistanceConnection
import os
import json

main_directory_path = str(os.getcwd())
# Path to the config file
config_path = main_directory_path + "\\utils\\config_partners_in_care.json"
# creating the config object to pull the authentication credentials later in various calls.
with open(config_path, "r", encoding="utf-8") as file:
    config_object = json.load(file)

# sql connection credentials
sql_username = config_object["Josh"]["username"]
sql_password = config_object["Josh"]["password"]
conn = GsAutomationReliefAssistanceConnection(sql_username, sql_password, auth=True)
conn.insert_with_progress(
    df=merge_df,
    schema="dbo",
    tbl_nm="partners_in_care_length_of_stay_and_utilization",
    dtypes={},
)